# E-matching is a relational join — laptop edition (CPU-only, runs anywhere)

Reproduces the core result of *Relational E-matching* (Zhang et al., POPL 2022) for `f(α, g(α))` with **no GPU and no compiler** — pure `pip` + Python, so it runs on any Windows / macOS / Linux laptop (or Colab).

Five verticals, each end-to-end and self-contained (no reuse) from the same raw *unsorted* relations:

| vertical | kind |
|---|---|
| Backtracking (nested loop) | O(N²) baseline |
| Hash join | O(N) |
| Leapfrog Triejoin | O(N), worst-case-optimal join |
| **DuckDB** | real engine, Arrow → SQL |
| **DataFusion** | real engine, Arrow → SQL (Rust/Arrow-native) |

It shows the same O(N) vs O(N²) separation as the GPU notebook, minus the GPU verticals. **Run all** (Jupyter / VS Code / Colab). Read **§5 Methodology** before trusting absolute numbers — the headline metric is the *slope*.

## 0. Parameters - edit these to scale the run.

In [ ]:
# ==== experiment parameters (edit these, then Run all) ====
MAX_N = 1 << 20    # O(N) verticals (hash / leapfrog / DuckDB / DataFusion). 1<<22 ~ 4M for a bigger run.
BT_CAP = 1 << 14   # backtracking is O(N^2) -> capped here (raising it gets slow fast).
print('MAX_N =', MAX_N, ' BT_CAP =', BT_CAP)

## 1. Install (pure pip — no compiler, no GPU)

In [ ]:
!pip -q install duckdb datafusion pyarrow numpy pandas matplotlib

## 2. Data, CPU matchers, and engine benches (all fresh per call — no reuse)

In [ ]:
import time, math
import numpy as np, pandas as pd, pyarrow as pa

I_F = 4_000_000_000_000_000_000
I_G = 5_000_000_000_000_000_000
SQL = ("SELECT f.id AS root, f.arg1 AS alpha FROM rf f JOIN rg g "
       "ON f.arg2 = g.id AND f.arg1 = g.arg1")

def gen_np(n, seed):
    # raw relations in RANDOM order (no method may free-ride on sortedness)
    rng = np.random.default_rng(seed)
    fk = np.arange(1, n + 1, dtype=np.int64); rng.shuffle(fk)
    gk = np.arange(1, n + 1, dtype=np.int64); rng.shuffle(gk)
    return (np.full(n, I_F, np.int64), fk, np.full(n, I_G, np.int64), np.full(n, I_G, np.int64), gk)

def gen_arrow(d):
    f_id, f_a1, f_a2, g_id, g_a1 = d
    return (pa.table({'id': pa.array(f_id), 'arg1': pa.array(f_a1), 'arg2': pa.array(f_a2)}),
            pa.table({'id': pa.array(g_id), 'arg1': pa.array(g_a1)}))

def best(fn, k):
    r = None; bt = 1e30
    for _ in range(k):
        t0 = time.perf_counter(); r = fn(); dt = time.perf_counter() - t0
        if dt < bt: bt = dt
    return bt, r

def check(rows, n):
    if len(rows) != n: return False
    seen = set()
    for r, a in rows:
        if r != I_F or not (1 <= a <= n) or a in seen: return False
        seen.add(a)
    return True

# ---- CPU matchers (pure NumPy/Python; each builds its own index) ----
def m_backtracking(d):                       # O(N^2)
    f_id, f_a1, f_a2, g_id, g_a1 = d; out = []
    for i in range(len(f_id)):
        m = (g_id == f_a2[i]) & (g_a1 == f_a1[i])
        for _ in range(int(m.sum())): out.append((int(f_id[i]), int(f_a1[i])))
    return out

def m_hashjoin(d):                           # O(N)
    f_id, f_a1, f_a2, g_id, g_a1 = d
    idx = set(zip(g_id.tolist(), g_a1.tolist())); out = []
    for r, a, x in zip(f_id.tolist(), f_a1.tolist(), f_a2.tolist()):
        if (x, a) in idx: out.append((r, a))
    return out

def _lb(a, lo, hi, v): return lo + int(np.searchsorted(a[lo:hi], v, 'left'))
def _ub(a, lo, hi, v): return lo + int(np.searchsorted(a[lo:hi], v, 'right'))
def m_lftj(d):                               # leapfrog triejoin: sort tries, gallop
    f_id, f_a1, f_a2, g_id, g_a1 = d
    fo = np.lexsort((f_id, f_a2, f_a1)); fa = f_a1[fo]; fx = f_a2[fo]; fr = f_id[fo]
    go = np.lexsort((g_id, g_a1)); ga = g_a1[go]; gx = g_id[go]
    n = len(fa); out = []; i = j = 0
    while i < n and j < n:
        av = int(fa[i]); ag = int(ga[j])
        if av < ag: i = _lb(fa, i + 1, n, ag); continue
        if ag < av: j = _lb(ga, j + 1, n, av); continue
        alpha = av; faHi = _ub(fa, i, n, alpha); gaHi = _ub(ga, j, n, alpha); p = i; q = j
        while p < faHi and q < gaHi:
            xf = int(fx[p]); xg = int(gx[q])
            if xf < xg: p = _lb(fx, p + 1, faHi, xg); continue
            if xg < xf: q = _lb(gx, q + 1, gaHi, xf); continue
            xv = xf; fxHi = _ub(fx, p, faHi, xv); r = p
            while r < fxHi:
                if r == p or fr[r - 1] != fr[r]: out.append((int(fr[r]), alpha))
                r += 1
            p = fxHi; q = _ub(gx, q, gaHi, xv)
        i = faHi; j = _ub(ga, j, n, alpha)
    return out

def bench_duckdb(d, n, reps=3):              # real engine; plan/exec split
    import duckdb
    rf, rg = gen_arrow(d); con = duckdb.connect(); con.register('rf', rf); con.register('rg', rg)
    full, rows = best(lambda: con.execute(SQL).fetchall(), reps)
    plan, _ = best(lambda: con.execute('EXPLAIN ' + SQL).fetchall(), reps); con.close()
    assert check(rows, n), 'duckdb wrong'
    return {'duckdb_exec_secs': max(full - plan, 0.0), 'duckdb_plan_secs': plan}

def bench_datafusion(d, n, reps=3):
    from datafusion import SessionContext
    rf, rg = gen_arrow(d); ctx = SessionContext()
    ctx.register_record_batches('rf', [rf.to_batches()]); ctx.register_record_batches('rg', [rg.to_batches()])
    plan, _ = best(lambda: ctx.sql(SQL), reps); df_ = ctx.sql(SQL)
    def run():
        out = []
        for b in df_.collect():
            dd = b.to_pydict(); out.extend(zip(dd['root'], dd['alpha']))
        return out
    exect, rows = best(run, reps); assert check(rows, n), 'datafusion wrong'
    return {'datafusion_exec_secs': exect, 'datafusion_plan_secs': plan}

## 3. Run the sweep
Edit `MAX_N` / `BT_CAP` in the Parameters cell (above). Backtracking is O(N²) so it stays capped; the O(N) verticals go to `MAX_N`.

In [ ]:
ns = [1 << e for e in range(10, MAX_N.bit_length())]
rows = []
for n in ns:
    d = gen_np(n, 1)                       # one raw input per N (shared INPUT, not reuse)
    rec = {'n': n, 'matches': n}
    t, r = best(lambda: m_hashjoin(d), 3); assert check(r, n); rec['hashjoin_secs'] = t
    t, r = best(lambda: m_lftj(d), 3);     assert check(r, n); rec['lftj_secs'] = t
    if n <= BT_CAP:
        t, r = best(lambda: m_backtracking(d), 2); assert check(r, n); rec['backtracking_secs'] = t
    rec.update(bench_duckdb(d, n)); rec.update(bench_datafusion(d, n))
    rows.append(rec); print('n =', n, 'done')
df = pd.DataFrame(rows)
df

## 4. Verification + asymptotic slopes

In [ ]:
assert (df['matches'] == df['n']).all(), 'match count != N somewhere'
print('correctness:', len(df), 'sizes, each exactly N matches (analytical ground truth).')
print()
VERTICALS = {
    'backtracking_secs':    ('Backtracking', 2, '#c0392b', 'o', '-'),
    'hashjoin_secs':        ('Hash join', 1, '#2980b9', 'o', '--'),
    'lftj_secs':            ('Leapfrog-TJ', 1, '#8e44ad', 'o', ':'),
    'duckdb_exec_secs':     ('DuckDB (exec)', 1, '#d35400', 'D', '-.'),
    'datafusion_exec_secs': ('DataFusion (exec)', 1, '#2c3e50', 'D', '-.'),
}
def fit_slope(s):
    s = s.sort_values('n')
    k = max(3, int(math.ceil(0.6 * len(s)))) if len(s) >= 3 else len(s)
    x = np.log(s['n'].to_numpy()[-k:]); y = np.log(s.iloc[:, 1].to_numpy()[-k:])
    sl, it = np.polyfit(x, y, 1)
    den = np.sum((y - y.mean())**2)
    return sl, (1 - np.sum((y - (sl*x+it))**2)/den if den > 0 else float('nan'))
fits = {}
print(f"{'vertical':<20}{'theory':>8}{'slope':>9}{'R2':>9}   verdict")
for col, (label, theory, *_ ) in VERTICALS.items():
    s = df[['n', col]].dropna(); s = s[s[col] > 0]
    if len(s) < 2: fits[col] = (float('nan'), label, theory); continue
    sl, r2 = fit_slope(s); fits[col] = (sl, label, theory)
    print(f"{label:<20}{('~N^'+str(theory)):>8}{sl:>9.3f}{r2:>9.4f}   {'OK' if abs(sl-theory)<0.3 else 'CHECK (need larger N?)'}")
print()
print('engine parse+plan overhead (tiny, constant in N; excluded from exec):')
print(df[['n','duckdb_plan_secs','datafusion_plan_secs']].dropna().tail(3).to_string(index=False))

## 4b. The diagram

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 11})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={'width_ratios': [1.6, 1]})
for col, (label, theory, c, m, ls) in VERTICALS.items():
    s = df[['n', col]].dropna(); s = s[s[col] > 0]
    if len(s) == 0: continue
    ax1.loglog(s['n'], s[col], marker=m, ls=ls, color=c, ms=5, lw=1.6, label=f"{label} (slope {fits[col][0]:.2f})")
ax1.set_xlabel('N (relation size)'); ax1.set_ylabel('join time [s] (min of reps)')
ax1.set_title('Backtracking vs relational joins (CPU only)')
ax1.grid(True, which='both', alpha=0.3); ax1.legend(fontsize=9, loc='upper left')
labels=[fits[c][1] for c in VERTICALS]; slopes=[fits[c][0] for c in VERTICALS]
theos=[fits[c][2] for c in VERTICALS]; cols=[VERTICALS[c][2] for c in VERTICALS]
yy=np.arange(len(labels))
ax2.barh(yy, slopes, color=cols, alpha=0.85)
ax2.scatter(theos, yy, color='black', marker='|', s=380, zorder=5, label='theory (1 or 2)')
ax2.axvline(1, color='gray', ls='--', lw=0.8); ax2.axvline(2, color='gray', ls='--', lw=0.8)
ax2.set_yticks(yy); ax2.set_yticklabels(labels, fontsize=9); ax2.invert_yaxis()
ax2.set_xlabel('measured exponent'); ax2.set_title('Empirical exponent vs theory'); ax2.set_xlim(0, 2.4)
ax2.grid(True, axis='x', alpha=0.3); ax2.legend(fontsize=8, loc='lower right')
fig.suptitle('E-matching is a join: relational O(N) vs backtracking O(N^2)', fontsize=13, y=1.02)
fig.tight_layout(); fig.savefig('ematching_laptop.png', dpi=140, bbox_inches='tight'); plt.show()
print('saved ematching_laptop.png')

## 5. Methodology & fairness

- **Headline metric: the slope.** A power law `time ∝ N^slope` is fit on each vertical's large-N region. Relational joins (hash, leapfrog, DuckDB, DataFusion) land near slope 1; backtracking near slope 2. The slope is robust to constant overheads — it *is* the paper's claim.
- **Each vertical is end-to-end and self-contained** from the same raw *unsorted* input: it builds its own hash table / sorts its own trie / parses its own query. Nothing derived is shared.
- **Correctness vs an independent ground truth:** the answer is known in closed form to be exactly `{(I_F,k):k=1..N}`; every vertical is checked against it.
- **Engine parse/plan overhead is excluded** from the engine `exec` columns (it is tiny and *constant* in N — shown separately). The hand-written matchers have the query effectively compiled in.
- **Hand-written vs engine:** the NumPy/Python baselines and the SQL engines differ by a large *constant* factor (interpreter vs vectorized engine, and the engines' generality) — the *exponent* is the fair comparison, not the absolute time.
- **Scope:** synthetic data isolating the worst case that separates backtracking from a join — not the egg/egglog suites. Single *acyclic* query, so leapfrog-TJ has no asymptotic edge over hash join here; its worst-case-optimal advantage needs a *cyclic* query (triangle). The GPU verticals (CUDA, Sirius) live in the GPU notebook.

## 6. Save results
Writes `results_laptop.csv` and `ematching_laptop.png` to the working directory (no Colab-specific download — portable).

In [ ]:
df.to_csv('results_laptop.csv', index=False)
import os
print('saved:', os.path.abspath('results_laptop.csv'))
print('saved:', os.path.abspath('ematching_laptop.png'))